In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import cv2 as cv
import numpy as np

girdi_video = '/content/drive/MyDrive/test_video.mp4'
cikti_video = '/content/drive/MyDrive/islenmis_video.mp4'

#videoyu okumak için
cap = cv.VideoCapture(girdi_video)

#orijinal özellikler
girdi_genislik = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
girdi_yukseklik = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
original_fps = cap.get(cv.CAP_PROP_FPS) #hız için

#hız ayarı- orijinal fps'i düşürürsen video yavaşlar, arttırırsan hızlanır.
yeni_fps = original_fps * 2

#yeni videoyu kaydetmek için hazırlık
fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter(cikti_video, fourcc, yeni_fps, (girdi_genislik, girdi_yukseklik))

print(f"Orijinal fps: {original_fps} | Yeni fps:{yeni_fps} ")
print("video işleniyor lütfen bekleyin...")

#Frame Frame işleme döngüsü
while True:
  ret,frame = cap.read() #her bir kareyi oku demek

  if not ret:
    break #eğer okunacak kare kalmadıysa döngüden çık video bittiyse yani

  #dikdörtgen çizimleri. ilk önce karenin ortasını bul.
  merkez_x, merkez_y = girdi_genislik // 2, girdi_yukseklik // 2
  #merkeze 100x100 boyutlarında, yeşil ve kalınlığı da 3 olan bir dikdörtgen çiziyoruz.
  cv.rectangle(frame, (merkez_x -50, merkez_y - 50), (merkez_x + 50, merkez_y + 50), (0,255,0), 3)

  #işlenmiş yeni kareyi çıktı videosuna yaz
  out.write(frame)

#dosyaları serbest bırak
cap.release()
out.release()
print(f"işlem tamamlandı yeni video drive'a kaydedildi: {cikti_video}")


Orijinal fps: 29.97002997002997 | Yeni fps:59.94005994005994 
video işleniyor lütfen bekleyin...
işlem tamamlandı yeni video drive'a kaydedildi: /content/drive/MyDrive/islenmis_video.mp4


In [ ]:
#thresholding yapılmış hali
import cv2 as cv
import numpy as np

girdi_video = '/content/drive/MyDrive/test_video.mp4'
cikti_video = '/content/drive/MyDrive/takip_edilen_drone.mp4'

cap = cv.VideoCapture(girdi_video)

#video özellikleri
genislik = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
yukseklik = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS) #hız için

#çıktı videosu için
fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter(cikti_video, fourcc, fps, (genislik, yukseklik))

print("renk tabanlı video başlatıldı lütfen bekleyin...")

while True:
  ret,frame = cap.read()
  if not ret:
    break

  #BGR formatını HSV formatına çeviriyoruz.
  hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

  #BEYAZ renk için alt ve üst sınırları belirle(maskeleme)
  #Beyaz rengin doygunluğu(S) düşük,parlaklığı(V) yüksektir.
  alt_beyaz = np.array([0,0,120])    #H:0 , S:0, V:180 (en az ne kadar beyaz)
  ust_beyaz = np.array([180,80,255]) #H:180, S:255, V:255 (en çok ne kadar beyaz)

  #bu sınırlar içindeki pikselleri beyaz(255),diğerlerini siyah(0) yapan maskeleme
  maske = cv.inRange(hsv_frame,alt_beyaz,ust_beyaz)

  #Dilation(genişletme) kısmı
  #15x15 boyutunda bir fırça oluşturup beyaz alanları şişiriyoruz.
  cekirdek = np.ones((15,15), np.uint8)
  maske = cv.dilate(maske, cekirdek, iterations=2)


  #maske üzerindeki beyaz alanların sınırlarını(konturlarını) bul
  konturlar, _ = cv.findContours(maske, cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)

  #eğer ekranda beyaz bir şey bulduysa:
  if len(konturlar) > 0:
    #en büyük beyaz alan=> drone bul
    en_buyuk_kontur = max(konturlar, key=cv.contourArea)

    #eğer bu alan çok küçük bir piksel hatası değilse(ufak bulut ya da kuş değilse)
    if cv.contourArea(en_buyuk_kontur) > 500:

      #droneun etrafını saracak en küçük dikdörtgen koordinatlarını al
      x,y, genislik_kutu, yukseklik_kutu = cv.boundingRect(en_buyuk_kontur)

      cv.rectangle(frame, (x, y), (x + genislik_kutu, y + yukseklik_kutu), (0,255,0), 3)
      cv.putText(frame, "DRONE", (x, y - 10), cv.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)

  out.write(frame)

cap.release()
out.release()
print(f"işlem tamamlandı yeni video drive'a kaydedildi: {cikti_video}")









renk tabanlı video başlatıldı lütfen bekleyin...
işlem tamamlandı yeni video drive'a kaydedildi: /content/drive/MyDrive/takip_edilen_drone.mp4


In [ ]:
#süre eklenmiş hali

import cv2 as cv
import numpy as np
from datetime import datetime

girdi_video = '/content/drive/MyDrive/test_video.mp4'
cikti_video = '/content/drive/MyDrive/takip_edilen_time1_drone.mp4'

cap = cv.VideoCapture(girdi_video)

genislik = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
yukseklik = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS) #hız için

fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter(cikti_video, fourcc, fps, (genislik, yukseklik))

print("renk tabanlı video başlatıldı lütfen bekleyin...")

ilk_tespit_zamani= None    #ilk görüldüğü anı hafızada tutucak

while True:
  ret,frame = cap.read()
  if not ret:
    break

  hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

  alt_beyaz = np.array([0,0,120])    #H:0 , S:0, V:180 (en az ne kadar beyaz)
  ust_beyaz = np.array([180,80,255]) #H:180, S:255, V:255 (en çok ne kadar beyaz)

  maske = cv.inRange(hsv_frame,alt_beyaz,ust_beyaz)

  cekirdek = np.ones((15,15), np.uint8)
  maske = cv.dilate(maske, cekirdek, iterations=2)

  konturlar, _ = cv.findContours(maske, cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)

  if len(konturlar) > 0:
    en_buyuk_kontur = max(konturlar, key=cv.contourArea)

    if cv.contourArea(en_buyuk_kontur) > 500:

      if ilk_tespit_zamani is None:  #eğer drone ilk defa görüldüyse o anı kaydet
        ilk_tespit_zamani = datetime.now()

      # şu anki sistem saatinden ilk tespit saatini çıkar.
      current_time = datetime.now()
      gecen_sure = (current_time - ilk_tespit_zamani).total_seconds()

      dakika = int(gecen_sure // 60)
      saniye = int(gecen_sure % 60)

      x,y, genislik_kutu, yukseklik_kutu = cv.boundingRect(en_buyuk_kontur)

      cv.rectangle(frame, (x, y), (x + genislik_kutu, y + yukseklik_kutu), (0,255,0), 3)
      gosterilecek_metin = f"DRONE: {dakika}:{saniye:02d}"

      cv.putText(frame, gosterilecek_metin, (x, y - 10), cv.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)

  out.write(frame)

cap.release()
out.release()
print(f"işlem tamamlandı yeni video drive'a kaydedildi: {cikti_video}")



renk tabanlı video başlatıldı lütfen bekleyin...
işlem tamamlandı yeni video drive'a kaydedildi: /content/drive/MyDrive/takip_edilen_time1_drone.mp4


In [ ]:
import cv2 as cv
import numpy as np

girdi_video = '/content/drive/MyDrive/coklu_drone.mp4'
cikti_video = '/content/drive/MyDrive/coklu_drone_takip4.mp4'

cap = cv.VideoCapture(girdi_video)

genislik = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
yukseklik = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv.CAP_PROP_FPS) #hız için

fourcc = cv.VideoWriter_fourcc(*'mp4v')
out = cv.VideoWriter(cikti_video, fourcc, fps, (genislik, yukseklik))

print("çoklu drone takibi başlatıldı lütfen bekleyin...")

toplam_kare_sayisi = 0
ilk_tespit_kare = None    #ilk görüldüğü anı hafızada tutucak

while True:
  ret,frame = cap.read()
  if not ret:
    break

  toplam_kare_sayisi += 1

  hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

  alt_siyah = np.array([0,0,0])
  ust_siyah = np.array([180,255,150]) #parlaklığı(V) 90 ın altında olan her şeyi siyah kabul et

  maske = cv.inRange(hsv_frame,alt_siyah,ust_siyah)

  cekirdek = np.ones((15,15), np.uint8)
  maske = cv.dilate(maske, cekirdek, iterations=2)

  konturlar, _ = cv.findContours(maske, cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)

  for kontur in konturlar:
    alan = cv.contourArea(kontur)
    #burda geometrik filtre yapıyorum. sadece belirli bir boyuttaki nesneleri al(çok küçük ya da devasa olanları alma)
    if  8000 < alan < 80000:
      x,y,genislik_kutu,yukseklik_kutu = cv.boundingRect(kontur)

      #en-boy oranı yapıyorum aspect ratio
      oran = float(genislik_kutu) / float(yukseklik_kutu)
      if 1.1 < oran < 3.5:

        if ilk_tespit_kare is None:  #eğer drone ilk defa görüldüyse o anı kaydet
            ilk_tespit_kare = toplam_kare_sayisi

        gecen_sure = (toplam_kare_sayisi - ilk_tespit_kare) / fps
        dakika = int(gecen_sure // 60)
        saniye = int(gecen_sure % 60)

        cv.rectangle(frame, (x, y), (x + genislik_kutu, y + yukseklik_kutu), (0,255,0), 3)

        debug_metin = f"A:{int(alan)} O:{oran:.1f}"
        cv.putText(frame, debug_metin, (x, y - 10), cv.FONT_HERSHEY_COMPLEX, 0.7, (0, 255, 0), 2)
  out.write(frame)
cap.release()
out.release()
print(f"işlem tamamlandı yeni video drive'a kaydedildi: {cikti_video}")

çoklu drone takibi başlatıldı lütfen bekleyin...
işlem tamamlandı yeni video drive'a kaydedildi: /content/drive/MyDrive/coklu_drone_takip4.mp4
